# Week 3 — Teach a machine to sort
*Albania Now — a Free Focus program, built with Chicago First. Run cells top to bottom. First: **File → Save a copy in Drive**.*

## 1. Labeled patches
Small crater and not-crater patches — plus labels, the machine's only source of truth.

In [ ]:
import numpy as np

def make_patch(kind, seed, size=15):
    rng = np.random.default_rng(seed)
    p = 175 + rng.integers(-20, 20, (size, size)).astype(float)
    yy, xx = np.mgrid[0:size, 0:size]
    c = size // 2
    if kind == "crater":
        d = np.sqrt((xx - c) ** 2 + (yy - c) ** 2)
        p[(d < 4.5) & (xx < c)] = 60
        p[(d < 4.5) & (xx >= c)] = 220
        p[(d >= 4.5) & (d < 5.5) & (xx > c)] = 238
    else:                       # a ridge shadow — dark but not round
        p[:, c - 1:c + 2] = 95 + rng.integers(-10, 10, (size, 3))
    return np.clip(p, 0, 255)

train = [(make_patch(k, s), k) for s, k in enumerate(
    ["crater", "ridge", "crater", "ridge", "crater", "ridge",
     "crater", "ridge", "crater", "ridge"])]
test  = [(make_patch(k, 100 + s), k) for s, k in enumerate(
    ["crater", "ridge", "ridge", "crater", "crater", "ridge"])]
print(len(train), "training patches,", len(test), "held-out patches")

## 2. Features — turn each patch into two numbers

In [ ]:
def features(p):
    dark = p < 110
    if not dark.any():
        return np.array([0.0, 0.0])
    ys, xs = np.nonzero(dark)
    spread_y = ys.std() + 1e-9
    spread_x = xs.std() + 1e-9
    roundness = min(spread_x, spread_y) / max(spread_x, spread_y)
    dark_frac = dark.mean()
    return np.array([roundness, dark_frac])

for p, k in train[:4]:
    print(k, features(p).round(2))

A crater's shadow is compact-round (roundness near 1); a ridge's is a stripe (roundness low). The feature already separates them — the machine just has to notice.

## 3. Train nearest-mean — the smallest real learner

In [ ]:
crater_mean = np.mean([features(p) for p, k in train if k == "crater"], axis=0)
ridge_mean  = np.mean([features(p) for p, k in train if k == "ridge"], axis=0)
print("crater mean:", crater_mean.round(2))
print("ridge  mean:", ridge_mean.round(2))

def classify(p):
    f = features(p)
    d_c = np.linalg.norm(f - crater_mean)
    d_r = np.linalg.norm(f - ridge_mean)
    return "crater" if d_c < d_r else "ridge"

## 4. Grade it — held-out only

In [ ]:
correct = sum(classify(p) == k for p, k in test)
print(f"Held-out score: {correct} of {len(test)}")

## 5. The two sabotages
**A.** Grade on training data and compare. **B.** Flip two training labels, retrain, re-grade on the held-out set.

In [ ]:
train_score = sum(classify(p) == k for p, k in train)
print(f"Score on its own training data: {train_score} of {len(train)}")
# B: flip two labels in `train`, rerun the training cell, and re-grade


## 6. The build
Report (5–8 sentences): the honest score, the two sabotage scores, and what each dishonesty teaches about machine learning claims you meet in the wild.

**Turn-in:** report + screenshots of the three scores.